In [1]:
// Data kilde
var tal = new List<int> { 1, 2, 3, 4, 5 };

// OPGAVE: Gør alle tal dobbelte og lav dem til strenge
// "Gammeldags" måde (Imperativ)
var resultatListe = new List<string>();
foreach(var t in tal) 
{
    resultatListe.Add($"Tallet er {t * 2}");
}

// LINQ måde (Deklarativ - "Hvad vil jeg have", ikke "hvordan")
// .Select = "Map" i andre sprog (transformerer 1 til 1)
var linqResultat = tal.Select(t => $"Tallet er {t * 2}").ToList();

// .Where = "Filter"
var kunStoreTal = tal.Where(t => t > 3).ToList(); // { 4, 5 }

Console.WriteLine(string.Join(", ", linqResultat));

Tallet er 2, Tallet er 4, Tallet er 6, Tallet er 8, Tallet er 10


## LINQ Deep Dive: Memory vs. Database
LINQ (Language Integrated Query) er måden, vi behandler data på.

### IEnumerable (Memory):

Bruges til lister, arrays, XML.

Koden kører i RAM.

Vi henter alt data, og filtrerer bagefter.

### IQueryable (Database / Remote):

Bruges til EF Core, SQL.

Koden er ikke kode! Det er et "Expression Tree" (en opskrift).

.NET oversætter din C# til SQL, før den sender det til databasen.

Deferred Execution (Udskudt kørsel): LINQ-queries kører IKKE, når du definerer dem. De kører først, når du "forbruger" dem (foreach, ToList, Count).

In [1]:
using System.Linq;

var numbers = new List<int> { 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 };

// 1. DEFERRED EXECUTION
Console.WriteLine("--- Deferred Execution ---");
// Vi definerer forespørgslen. Intet sker her!
var query = numbers.Where(n => 
{
    Console.WriteLine($"Tjekker tal: {n}"); // Denne linje beviser hvornår det kører
    return n % 2 == 0;
});

Console.WriteLine("Query er defineret. Nu kører vi ToList()...");
// Først NU kører logikken ovenover
var evenNumbers = query.ToList(); 
Console.WriteLine($"Fandt {evenNumbers.Count} lige tal.");


// 2. IENUMERABLE VS IQUERYABLE (Simuleret)
// I din Sommerhus app bliver dette kritisk.

// Scenario: Vi har 1.000.000 ordrer i databasen.
// IQueryable<Order> dbOrders = _context.Orders;

// GODT (IQueryable):
// dbOrders.Where(o => o.Total > 1000).ToList();
// -> SQL: SELECT * FROM Orders WHERE Total > 1000
// -> Vi henter kun de relevante rækker.

// SKIDT (IEnumerable / Memory Leak):
// dbOrders.ToList().Where(o => o.Total > 1000).ToList();
// -> SQL: SELECT * FROM Orders (Henter 1 million rækker til RAM!)
// -> C#: Filtrerer dem bagefter. Serveren crasher.


// 3. AVANCERET: SELECT MANY & GROUP BY
// Use Case: Sommerhuse. Vi vil finde alle features på tværs af alle huse i et område.
record Feature(string Name);
record House(string Name, List<Feature> Features);

var houses = new List<House> 
{
    new House("Villa Sol", new() { new("Pool"), new("WiFi") }),
    new House("Hytten",    new() { new("WiFi"), new("Pejs") })
};

// SelectMany "flader ud" (fra liste af lister -> til én lang liste)
var allFeatures = houses
    .SelectMany(h => h.Features) // Giver os alle features i én pærevælling
    .Select(f => f.Name)
    .Distinct() // Fjerner dubletter (WiFi var der to gange)
    .ToList();

Console.WriteLine("Unikke features: " + string.Join(", ", allFeatures));

--- Deferred Execution ---
Query er defineret. Nu kører vi ToList()...
Tjekker tal: 1
Tjekker tal: 2
Tjekker tal: 3
Tjekker tal: 4
Tjekker tal: 5
Tjekker tal: 6
Tjekker tal: 7
Tjekker tal: 8
Tjekker tal: 9
Tjekker tal: 10
Fandt 5 lige tal.
Unikke features: Pool, WiFi, Pejs


 ### Under Motorhjelmen – Hvordan virker LINQ egentlig?
Mange bruger LINQ som magi. Lad os fjerne magien og bygge det selv. LINQ er baseret på Extension Methods og Iterators (yield).

For at forstå Deferred Execution (udskudt kørsel) til bunds, skal vi se, hvordan Where ser ud, hvis vi selv skulle kode den.

(Illustration af hvordan data flyder igennem en iterator én ad gangen)

In [ ]:
using System.Collections.Generic;

// VI BYGGER VORES EGEN LINQ OPERATOR!
public static class MinEgenLinq
{
    // Dette er en "Extension Method" (this keywordet gør tricket)
    public static IEnumerable<T> MitFilter<T>(this IEnumerable<T> kilde, Func<T, bool> test)
    {
        Console.WriteLine("--> MitFilter starter (Men looper ikke endnu!)");
        
        foreach (var item in kilde)
        {
            Console.WriteLine($"--> MitFilter kigger på: {item}");
            if (test(item))
            {
                // YIELD RETURN er hemmeligheden bag Deferred Execution!
                // Den returnerer ét element og PAUSER funktionen her.
                yield return item; 
            }
        }
        Console.WriteLine("--> MitFilter færdig");
    }
}

var tal = new List<int> { 1, 2, 3, 4, 5 };

Console.WriteLine("1. Opsætter query...");
// Bemærk: Vi kalder vores egen metode som om den var en del af listen
var query = tal.MitFilter(x => x > 2); 

Console.WriteLine("2. Query opsat. Intet er kørt endnu.");
Console.WriteLine("3. Starter foreach nu:");

// Prøv at køre dette og se loggen nøje. 
// Læg mærke til at den skifter mellem "kigger på" og at udskrive tallet.
foreach(var t in query)
{
    Console.WriteLine($"   Modtog tal: {t}");
}

### Modul 2: Det store opslagsværk (Kategorier)
LINQ har over 50 metoder. Her er de vigtigste opdelt i kategorier, som du skal kende.

Indsæt denne Markdown/Code blok:

1. Projektion (Forme data)

Select: 1-til-1 transformation.

SelectMany: 1-til-Mange (Flader lister ud).

In [6]:
var hold = new[] 
{
    new { Navn = "A-klassen", Elever = new[] { "Ib", "Bo" } },
    new { Navn = "B-klassen", Elever = new[] { "Lis", "Ann" } }
};

// Select bevarer strukturen (Liste af Lister)
var selectResult = hold.Select(h => h.Elever); 
Console.WriteLine($"Select count: {selectResult.Count()} (Det er 2 lister)");

// SelectMany flader det ud til én lang liste af elever
var alleElever = hold.SelectMany(h => h.Elever);
Console.WriteLine($"SelectMany count: {alleElever.Count()} (Det er 4 elever): " + string.Join(", ", alleElever));

Select count: 2 (Det er 2 lister)
SelectMany count: 4 (Det er 4 elever): Ib, Bo, Lis, Ann


2. Set Operations (Mængdelære)
Når du skal sammenligne to lister.

In [7]:
var liste1 = new[] { 1, 2, 3, 4 };
var liste2 = new[] { 3, 4, 5, 6 };

// Intersect: Hvad har de til fælles?
var fælles = liste1.Intersect(liste2); // { 3, 4 }

// Except: Hvad har liste 1, som liste 2 IKKE har?
var kunIEn = liste1.Except(liste2); // { 1, 2 }

// Union: Alt fra begge (uden dubletter)
var alle = liste1.Union(liste2); // { 1, 2, 3, 4, 5, 6 }

Console.WriteLine($"Fælles: {string.Join(",", fælles)} | Kun i 1: {string.Join(",", kunIEn)}");

Fælles: 3,4 | Kun i 1: 1,2


3. Aggregering (Matematik)
Count, Sum, Min, Max, Average.

Aggregate: Den mest kraftfulde (og sværeste). Den "folder" en liste sammen.

In [8]:
int[] numre = { 1, 2, 3, 4 };

// Simpel
var sum = numre.Sum();

// Aggregate (Advanced): Byg din egen sum, eller byg en streng
// Startværdi: 0. 
// (acc, next) => acc + next betyder: Tag totalen hidtil (acc) og læg næste tal til.
var manuelSum = numre.Aggregate(0, (total, næsteTal) => total + næsteTal);

// Aggregate string builder
var kommaListe = numre.Aggregate("Tal:", (str, n) => str + " [" + n + "]");
Console.WriteLine(kommaListe); // Output: Tal: [1] [2] [3] [4]

Tal: [1] [2] [3] [4]


### Modul 3: Grouping og Lookup (VIGTIGT!)
Dette er det sværeste for begyndere, men essentielt for business apps.

In [9]:
record Produkt(string Kategori, string Navn, decimal Pris);

var produkter = new List<Produkt>
{
    new("Frugt", "Æble", 5),
    new("Frugt", "Pære", 6),
    new("Grønt", "Gulerod", 4),
    new("Grønt", "Kartoffel", 8),
    new("Slik", "Chokolade", 20)
};

// GROUP BY: Laver en liste af "Key" og en liste af items under den key
var grupper = produkter.GroupBy(p => p.Kategori);

foreach(var gruppe in grupper)
{
    Console.WriteLine($"Kategori: {gruppe.Key} (Antal varer: {gruppe.Count()})");
    foreach(var vare in gruppe)
    {
        Console.WriteLine($" - {vare.Navn}: {vare.Pris} kr");
    }
}

// TO LOOKUP: Ligner GroupBy, men sker med det samme (Immediate Execution)
// Perfekt til caching.
var lookup = produkter.ToLookup(p => p.Kategori);
// Nu kan vi lynhurtigt slå op uden at iterere hele listen:
var alleFrugter = lookup["Frugt"];

Kategori: Frugt (Antal varer: 2)
 - Æble: 5 kr
 - Pære: 6 kr
Kategori: Grønt (Antal varer: 2)
 - Gulerod: 4 kr
 - Kartoffel: 8 kr
Kategori: Slik (Antal varer: 1)
 - Chokolade: 20 kr


### Modul 4: IQueryable & Expression Trees (Sort Magi)
I din oprindelige notebook nævnte du forskellen på Memory og Database. Her beviser vi, at IQueryable ikke er kode, men data!

Når du skriver LINQ til en database, oversætter compileren din lambda x => x > 5 til et Expression Tree, som SQL-driveren kan læse og lave om til SQL (WHERE x > 5).

In [10]:
using System.Linq.Expressions;

// Vi laver en Expression manuelt (dette gør compileren normalt for dig)
Expression<Func<int, bool>> expr = num => num > 5;

// Lad os inspicere "koden" som om det var data
Console.WriteLine($"Hele udtrykket: {expr}");
Console.WriteLine($"Body (logikken): {expr.Body}");
Console.WriteLine($"Parameter: {expr.Parameters[0].Name}");

var binaryExpr = (BinaryExpression)expr.Body;
Console.WriteLine($"Venstre side: {binaryExpr.Left}");
Console.WriteLine($"Operator: {binaryExpr.NodeType}");
Console.WriteLine($"Højre side: {binaryExpr.Right}");

// Dette beviser, at Entity Framework kan "læse" din kode og oversætte den til SQL
// uden nogensinde at køre C# koden mod dataene.

Hele udtrykket: num => (num > 5)
Body (logikken): (num > 5)
Parameter: num
Venstre side: num
Operator: GreaterThan
Højre side: 5


Modul 5: Den Store Opgave – "Supermarkedet"
Her er en interaktiv sektion, hvor brugeren skal løse problemer.

In [ ]:
// --- DATA SETUP ---
public record Kunde(int Id, string Navn, int Alder);
public record Ordre(int Id, int KundeId, DateTime Dato, decimal TotalBeløb, List<OrdreLinje> Linjer);
public record OrdreLinje(string Vare, int Antal, decimal StkPris);

var kunder = new List<Kunde> 
{
    new(1, "Jens", 35), new(2, "Mette", 28), new(3, "Ole", 72), new(4, "Pia", 45)
};

var ordrer = new List<Ordre>
{
    new(101, 1, new DateTime(2023, 1, 15), 500, new() { new("Mælk", 2, 10), new("Ost", 1, 80) }),
    new(102, 1, new DateTime(2023, 2, 01), 1200, new() { new("Vin", 6, 100), new("Chips", 5, 20) }),
    new(103, 2, new DateTime(2023, 1, 20), 150, new() { new("Mælk", 1, 10), new("Brød", 2, 25) }),
    new(104, 3, new DateTime(2023, 3, 10), 0, new() { }) // Annulleret ordre
};

// --- DINE OPGAVER (Udfyld koden under hver) ---

// OPGAVE 1: Find alle kunder over 40 år
// Hint: Where
// var opgave1 = ...;
// Console.WriteLine("Opgave 1: " + string.Join(", ", opgave1.Select(k => k.Navn)));

// OPGAVE 2: Find summen af alle ordrer (TotalBeløb) lagt i januar måned (Måned 1)
// Hint: Where + Sum
// var opgave2 = ...;
// Console.WriteLine($"Opgave 2: {opgave2} kr.");

// OPGAVE 3: Hvilken vare er solgt flest gange (Antal) på tværs af alle ordrer?
// Hint: SelectMany -> GroupBy -> OrderByDescending -> First
// var opgave3 = ...;
// Console.WriteLine($"Opgave 3: Mest solgte vare er ...");

// OPGAVE 4: Lav en liste af strenge der viser: "KundeNavn har købt for X kr i alt"
// Hint: Join (Kunder og Ordrer) eller GroupJoin
// var opgave4 = ...;